# RCA Knowledge Graph — Exploration

Quick-iterate notebook: drive the same pipeline used by the CLI, but interactively.
Make sure `.env` is configured (especially `ANTHROPIC_API_KEY`) before running.

In [ ]:
import sys, asyncio
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from rca_knowledge.config import load_settings
from rca_knowledge.ingestion.pipeline import IngestionPipeline
from rca_knowledge.reasoning.causal_query import CausalReasoner

settings = load_settings()
pipeline = IngestionPipeline(settings)
reasoner = CausalReasoner(settings)
settings.extraction_model, settings.reasoning_model

## 1. Ingest some text

In [ ]:
sample = '''
Chemical-mechanical planarization (CMP) of copper damascene structures
exhibits dishing — a downward bowing of wide Cu lines — when downforce
and slurry abrasive concentration are too high. Dishing thins the metal
cross-section and degrades barrier (Ta/TaN) integrity at the line edge,
increasing via and line resistance and accelerating electromigration
failure under operating current density.
'''
results = await pipeline.ingest_text(sample, source_label='cmp-intro', run_cognify=True)
for r in results:
    print(r.source_label, len(r.extraction.entities), len(r.extraction.relations))
    print(r.extraction.summary)

## 2. Run a causal query

In [ ]:
assessment = await reasoner.assess(
    'CMP downforce correlates with via resistance at r=0.68',
    process_context='Cu damascene, M2, 28nm node',
)
print(assessment.verdict)
print(assessment.verdict_reasoning)
for m in assessment.mechanisms:
    print(' -', m.confidence, '|', m.description)
for c in assessment.confounders:
    print(' confounder:', c.common_cause, '-', c.description)

## 3. Run the benchmark

In [ ]:
from rca_knowledge.evaluation.benchmark import run_benchmark
report = await run_benchmark(settings, ROOT / 'data' / 'benchmark' / 'test_cases.yaml')
report.summary